## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
experiment_ids = input_experiment_ids()

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
calib_input = get_input_with_default(
    "Do you want to use new calibration? [y/n, or press Enter for yes]",
    "y",
    str
)

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
settings = get_nasa_generation_settings(calib_key)
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

## Pulse Height Distribution

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    
    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    neutrons_only = psd_report.query(n_class_col_name).copy()
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    signals_df = exp_data["signals_df"]
    n_signals_df = signals_df.loc[neutrons_only.index].astype("uint32")

    n_signals_np = n_signals_df.to_numpy()
    baselines = n_signals_np.max(axis=1).reshape(-1, 1)
    n_signals_np = -n_signals_np + baselines
    n_signals_df = pd.DataFrame(n_signals_np, index=n_signals_df.index, columns=n_signals_df.columns)
    neutrons_only["peak_height"] = n_signals_df.max(axis=1)
    print(neutrons_only["peak_height"].max())
    print(neutrons_only["peak_height"].min())
    
    exp_data["n_signals_df"] = n_signals_df
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    peak_height = neutrons_only["peak_height"]
    # print(neutron_energies.max())
    # print(neutron_energies.min())

    energy_bins = np.arange(0, 12000, step=200)
    
    Z_n, *_ = np.histogram(peak_height, bins=energy_bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        "neutron": {"standard": Z_n, "bins": energy_bins},
    }

In [ ]:
# moving average
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_n_histogram = phd_histogram_data["neutron"]["standard"]
    # phd_g_histogram = phd_histogram_data["gamma"]["standard"]

    phd_n_moving_average = moving_average_centered(phd_n_histogram)
    # phd_g_moving_average = moving_average_centered(phd_g_histogram)

    phd_histogram_data["neutron"]["moving_average"] = phd_n_moving_average
    # phd_histogram_data["gamma"]["moving_average"] = phd_g_moving_average
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_histogram_data

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    # energy_bins = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]
    n_signals_df = exp_data["n_signals_df"]
    energy_bins = phd_histogram_data["bins"]

    energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
    trace_bin_lo = energy_bins[::5][1:]
    trace_bin_hi = energy_bins[1::5][1:]
    trace_bins = list(zip(trace_bin_lo, trace_bin_hi))
    traces = []
    for trace_bin in trace_bins:
        bin_lo, bin_hi = trace_bin
        bin_mid = (bin_lo + bin_hi) / 2
        matching_neutrons = neutrons_only[neutrons_only["peak_height"].between(bin_lo, bin_hi)]
        if len(matching_neutrons) == 0:
            continue
        # TODO search for traces without dual peaks
        for neutron_id in matching_neutrons.index:
            matching_trace = n_signals_df.loc[neutron_id]
            
            peaks, peak_data = find_peaks(matching_trace, height=200, prominence=50)
            filtered_peaks = [peak for peak in peaks if abs(peak - 50) > 15]
            if len(filtered_peaks) == 0:
                traces.append((bin_mid, matching_trace))
                break
    traces = sample(traces, len(traces))

    exp_data["selected_traces"] = traces

## Display

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]
    selected_traces = exp_data["selected_traces"]
    energy_bins = phd_histogram_data["bins"]
    phd_n_histogram = phd_histogram_data["standard"]
    phd_n_moving_avg = phd_histogram_data["moving_average"]
    phd_n_moving_avg = np.nan_to_num(phd_n_moving_avg)

    energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2

    phd_spline = CubicSpline(energy_bin_mids, phd_n_moving_avg)
    phd_spline_x = np.linspace(0, energy_bins.max(), num=1000)
    phd_spline_y = phd_spline(phd_spline_x)

    fig, axs = plt.subplots(ncols=2, figsize=(12, 8))
    fig.subplots_adjust(wspace=0)
    ax1, ax2 = axs

    ax1.plot(phd_spline_y, phd_spline_x, color="black")
    for i, (bin_mid, trace) in enumerate(selected_traces):
        trace_x = [x + i * 25 for x in range(len(trace))]
        ax2.plot(trace_x, trace, label=bin_mid)

    ax1.xaxis.set_inverted(True)
    ax1.xaxis.set_tick_params(labelbottom=False)
    ax1.yaxis.set_tick_params(labelbottom=False, bottom=False)
    ax2.xaxis.set_tick_params(labelbottom=False)
    ax2.yaxis.set_tick_params(labelbottom=False, direction="in")
    
    ax1.spines["top"].set_visible(False)
    ax1.spines["left"].set_visible(False)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)

    ax1.set_xlabel("Counts", fontsize=fontsize)
    ax2.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax2.set_xlabel("Time (ns)", fontsize=fontsize)

    limits = (0, 5000)
    ax1.set_ylim(*limits)
    ax2.set_ylim(*limits)
    ax1.set_xlim(None, 5)
    ax2.set_xlim(None, 200)

    for i, (bin_mid, trace) in enumerate(selected_traces):
        bin_mid_idx = np.where(energy_bin_mids == bin_mid)[0]
        bin_lo = float(energy_bins[bin_mid_idx][0])
        bin_hi = float(energy_bins[bin_mid_idx+1][0])
        bin_phd_x = np.linspace(bin_lo, bin_hi).reshape(-1, 1)
        bin_phd_y = phd_spline(bin_phd_x).reshape(-1, 1)
        bin_phd_xy = np.concatenate((bin_phd_y, bin_phd_x), axis=1)
        trace_max_x = float(trace.idxmax()) + (i * 25)
        bin_trace_xy = [[trace_max_x, bin_hi], [trace_max_x, bin_lo]]
        
        ax1_to_display = ax1.transData.transform
        ax2_to_display = ax2.transData.transform
        display_to_figure = fig.transFigure.inverted().transform

        bin_phd_xy = display_to_figure(ax1_to_display(bin_phd_xy))
        bin_trace_xy = display_to_figure(ax2_to_display(bin_trace_xy))
        bin_xy = np.concatenate((bin_phd_xy, bin_trace_xy))
        
        poly = mpl.patches.Polygon(bin_xy, closed=True, color="lightblue", alpha=0.3)
        fig.add_artist(poly)

    # get bin midpoints
    # make subplots (2 cols, 1 row, merged y-axis, no borders)
    # plot PHD histogram on left subplot (rotated)
    # pick 5 evenly spaced bins
    # for each bin, find first trace within that bin
    # (alternate, scan bin for trace with energy closest to midpoint of bin)
    # plot 5 traces on right subplot
    # make connecting lines between left subplot (bin midpoint, bin count) and right subplot (bin midpoint, trace height)

In [ ]:
input("Processing done, hit Enter to finish")
stop()